<a href="https://colab.research.google.com/github/praveenvadde1250/trendpulse-vaddepraveen/blob/main/task2_data_processing_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import requests
import time
import json
import os
from datetime import datetime

# ---------------------------------------------
# CONFIGURATION
# ---------------------------------------------

BASE_URL = "https://hacker-news.firebaseio.com/v0"
HEADERS = {"User-Agent": "TrendPulse/1.0"}
MAX_STORIES_PER_CATEGORY = 25

CATEGORIES = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

# ---------------------------------------------
# FUNCTIONS
# ---------------------------------------------

def fetch_top_story_ids():
    try:
        url = f"{BASE_URL}/topstories.json"
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        return response.json()[:500]
    except Exception as e:
        print(f"Error fetching top stories: {e}")
        return []


def fetch_story_details(story_id):
    try:
        url = f"{BASE_URL}/item/{story_id}.json"
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching story {story_id}: {e}")
        return None


def assign_category(title):
    if not title:
        return None

    title = title.lower()

    for category, keywords in CATEGORIES.items():
        for keyword in keywords:
            if keyword in title:
                return category

    return None


# ---------------------------------------------
# MAIN EXECUTION
# ---------------------------------------------

def main():
    story_ids = fetch_top_story_ids()

    if not story_ids:
        print("No stories fetched. Exiting...")
        return

    collected_stories = []
    category_count = {cat: 0 for cat in CATEGORIES}

    # Loop through each category
    for category in CATEGORIES:
        print(f"Processing category: {category}")

        for story_id in story_ids:

            if category_count[category] >= MAX_STORIES_PER_CATEGORY:
                break

            story = fetch_story_details(story_id)

            if not story:
                continue

            title = story.get("title", "")
            detected_category = assign_category(title)

            if detected_category == category:

                story_data = {
                    "post_id": story.get("id"),
                    "title": title,
                    "category": category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0),
                    "author": story.get("by", "unknown"),
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }

                collected_stories.append(story_data)
                category_count[category] += 1

        # Required delay after each category
        time.sleep(2)

    # ---------------------------------------------
    # SAVE TO JSON
    # ---------------------------------------------

    os.makedirs("data", exist_ok=True)

    today_str = datetime.now().strftime('%Y%m%d')
    filename = f"data/trends_{today_str}.json"

    with open(filename, "w", encoding="utf-8") as file:
        json.dump(collected_stories, file, indent=4)

    # ---------------------------------------------
    # FINAL OUTPUT (MATCHES REQUIREMENT EXACTLY)
    # ---------------------------------------------

    print(f"Collected {len(collected_stories)} stories. Saved to {filename}")


# ---------------------------------------------
# RUN SCRIPT
# ---------------------------------------------

if __name__ == "__main__":
    main()

Processing category: technology
Processing category: worldnews
Processing category: sports
Processing category: science
Processing category: entertainment
Collected 84 stories. Saved to data/trends_20260413.json


### Step 1: Set up your GitHub Personal Access Token (PAT)

To push to GitHub, you'll need a Personal Access Token (PAT). If you don't have one, you can generate it from your GitHub settings (Settings > Developer settings > Personal access tokens).

**Important:** For security reasons, store your PAT in Colab's secrets manager. Click the '🔑' icon on the left sidebar, add a new secret, and name it `GH_TOKEN`. Paste your PAT as the value. Make sure 'Notebook access' is enabled for this secret.

In [14]:
# Import necessary libraries
from google.colab import userdata
import os
import shutil
import re # Import regex for repository name extraction

# Configure Git with your user name and email
# Replace with your actual GitHub username and email
!git config --global user.name "praveenvadde1250"
!git config --global user.email "praveen.vadde1250@gmail.com"

# Get GitHub token from Colab secrets
GH_TOKEN = userdata.get('GH_TOKEN')

# --- IMPORTANT: Provide your GitHub repository URL here ---
# It should look like "https://github.com/your-username/your-repo-name.git"
github_repo_url = "https://github.com/praveenvadde1250/trendpulse-vaddepraveen"

# Extract repository name from the URL to use as local directory name
# Example: from 'https://github.com/user/repo.git' extracts 'repo'
match = re.search(r'/([^/]+?)(?:\.git)?$', github_repo_url)
if match:
    repo_name = match.group(1)
else:
    # Fallback if URL format is unexpected
    repo_name = "trendpulse-vaddepraveen"

repo_dir = repo_name

# Construct the URL with token for cloning
repo_with_token = github_repo_url.replace("https://", f"https://oauth2:{GH_TOKEN}@")

# Remove existing local repo directory if it exists, for a clean clone
if os.path.exists(repo_dir):
    print(f"Removing existing directory '{repo_dir}'...")
    shutil.rmtree(repo_dir)

# Clone the repository
print(f"Cloning {github_repo_url} into '{repo_dir}'...")
!git clone {repo_with_token} {repo_dir}

# Save the content of the main script cell to a file
main_script_content = """
import requests
import time
import json
import os
from datetime import datetime

# ---------------------------------------------
# CONFIGURATION
# ---------------------------------------------

BASE_URL = "https://hacker-news.firebaseio.com/v0"
HEADERS = {"User-Agent": "TrendPulse/1.0"}
MAX_STORIES_PER_CATEGORY = 25

CATEGORIES = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

# ---------------------------------------------
# FUNCTIONS
# ---------------------------------------------

def fetch_top_story_ids():
    try:
        url = f"{BASE_URL}/topstories.json"
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        return response.json()[:500]
    except Exception as e:
        print(f"Error fetching top stories: {e}")
        return []


def fetch_story_details(story_id):
    try:
        url = f"{BASE_URL}/item/{story_id}.json"
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching story {story_id}: {e}")
        return None


def assign_category(title):
    if not title:
        return None

    title = title.lower()

    for category, keywords in CATEGORIES.items():
        for keyword in keywords:
            if keyword in title:
                return category

    return None


# ---------------------------------------------
# MAIN EXECUTION
# ---------------------------------------------

def main():
    story_ids = fetch_top_story_ids()

    if not story_ids:
        print("No stories fetched. Exiting...")
        return

    collected_stories = []
    category_count = {cat: 0 for cat in CATEGORIES}

    # Loop through each category
    for category in CATEGORIES:
        print(f"Processing category: {category}")

        for story_id in story_ids:

            if category_count[category] >= MAX_STORIES_PER_CATEGORY:
                break

            story = fetch_story_details(story_id)

            if not story:
                continue

            title = story.get("title", "")
            detected_category = assign_category(title)

            if detected_category == category:

                story_data = {
                    "post_id": story.get("id"),
                    "title": title,
                    "category": category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0),
                    "author": story.get("by", "unknown"),
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }

                collected_stories.append(story_data)
                category_count[category] += 1

        # Required delay after each category
        time.sleep(2)

    # ---------------------------------------------
    # SAVE TO JSON
    # ---------------------------------------------

    os.makedirs("data", exist_ok=True)

    today_str = datetime.now().strftime('%Y%m%d')
    filename = f"data/trends_{today_str}.json"

    with open(filename, "w", encoding="utf-8") as file:
        json.dump(collected_stories, file, indent=4)

    # ---------------------------------------------
    # FINAL OUTPUT (MATCHES REQUIREMENT EXACTLY)
    # ---------------------------------------------

    print(f"Collected {len(collected_stories)} stories. Saved to {filename}")


# ---------------------------------------------
# RUN SCRIPT
# ---------------------------------------------

if __name__ == "__main__":
    main()"""

with open("task1_data_collection.py", "w") as f:
    f.write(main_script_content)

# Copy the main script to the new repository directory
shutil.copy("task1_data_collection.py", os.path.join(repo_dir, "task1_data_collection.py"))

# Copy the data folder to the new repository directory
# Remove existing data folder in the cloned repo to replace with new one
existing_data_path_in_repo = os.path.join(repo_dir, "data")
if os.path.exists(existing_data_path_in_repo):
    print(f"Removing existing data folder in '{repo_dir}'...")
    shutil.rmtree(existing_data_path_in_repo)

if os.path.exists("data"): # Check if the original data folder exists
    print(f"Copying new data folder to '{repo_dir}'...")
    shutil.copytree("data", existing_data_path_in_repo)
else:
    print("No 'data' folder found to copy.")

# Change to the repository directory
%cd {repo_dir}

# Add all changes (new/modified files) to the staging area
!git add .

# Commit the changes
# Using --allow-empty-message and -m '' to commit if no changes (e.g., if only script was updated and data was the same)
!git commit -m "Update: HackerNews TrendPulse script and data" --allow-empty

print(f"Git repository updated and files committed in '{repo_dir}'")

Cloning https://github.com/praveenvadde1250/trendpulse-vaddepraveen into 'trendpulse-vaddepraveen'...
Cloning into 'trendpulse-vaddepraveen'...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 5 (delta 0), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), 6.76 KiB | 6.76 MiB/s, done.
Removing existing data folder in 'trendpulse-vaddepraveen'...
Copying new data folder to 'trendpulse-vaddepraveen'...
/content/trendpulse-vaddepraveen/trendpulse-vaddepraveen/trendpulse-vaddepraveen/trendpulse-vaddepraveen
[main ed588c3] Update: HackerNews TrendPulse script and data
 1 file changed, 137 insertions(+)
 create mode 100644 task1_data_collection.py
Git repository updated and files committed in 'trendpulse-vaddepraveen'


### Step 2: Push to GitHub

Now, provide your GitHub repository URL. It should look like `https://github.com/your-username/your-repo-name.git`. This will be used to push your code.

In [15]:
# Push the changes to the main branch
# The remote 'origin' is already set by the git clone command in the previous cell.
# The GH_TOKEN is handled during the clone operation.

# It's important to be in the correct directory (which should be handled by the previous cell's %cd)
!git push -u origin main

print("Code and data pushed to GitHub!")

Enumerating objects: 3, done.
Counting objects: 100% (3/3), done.
Delta compression using up to 2 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (2/2), 324 bytes | 324.00 KiB/s, done.
Total 2 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/praveenvadde1250/trendpulse-vaddepraveen
   25cf609..ed588c3  main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.
Code and data pushed to GitHub!


In [16]:
import pandas as pd
import os
from datetime import datetime

# ---------------------------------------------
# STEP 1 — LOAD JSON FILE
# ---------------------------------------------

# Find latest JSON file in data/ folder
data_folder = "data"
json_files = [f for f in os.listdir(data_folder) if f.startswith("trends_") and f.endswith(".json")]

if not json_files:
    print("No JSON files found in data/ folder.")
    exit()

# Pick latest file
latest_file = sorted(json_files)[-1]
file_path = os.path.join(data_folder, latest_file)

# Load into DataFrame
df = pd.read_json(file_path)

print(f"Loaded {len(df)} stories from {file_path}")

# ---------------------------------------------
# STEP 2 — CLEAN THE DATA
# ---------------------------------------------

# 1. Remove duplicates
df = df.drop_duplicates(subset=["post_id"])
print(f"After removing duplicates: {len(df)}")

# 2. Remove missing values
df = df.dropna(subset=["post_id", "title", "score"])
print(f"After removing nulls: {len(df)}")

# 3. Fix data types
df["score"] = df["score"].astype(int)
df["num_comments"] = df["num_comments"].astype(int)

# 4. Remove low-quality stories (score < 5)
df = df[df["score"] >= 5]
print(f"After removing low scores: {len(df)}")

# 5. Strip whitespace from title
df["title"] = df["title"].str.strip()

# ---------------------------------------------
# STEP 3 — SAVE AS CSV
# ---------------------------------------------

output_file = os.path.join(data_folder, "trends_clean.csv")
df.to_csv(output_file, index=False)

print(f"\nSaved {len(df)} rows to {output_file}")

# ---------------------------------------------
# SUMMARY — STORIES PER CATEGORY
# ---------------------------------------------

print("\nStories per category:")
print(df["category"].value_counts())

Loaded 84 stories from data/trends_20260413.json
After removing duplicates: 84
After removing nulls: 84
After removing low scores: 83

Saved 83 rows to data/trends_clean.csv

Stories per category:
category
technology       25
entertainment    25
sports           14
worldnews        13
science           6
Name: count, dtype: int64
